# Analysis of neighboring pixels

This notebook is for investigating how model performance changes if we extend the input features to include neighboring pixels as well as the one we are trying to predict. My hope is that these models will perform significantly better than the previous models I have made.

In [10]:
import pyarrow  # must be imported before torch to avoid ArrowKeyError on read_parquet
import pandas as pd
import glob
import numpy as np

PARQUET_DIR = "C:\\Users\\admin\\Documents\\Glacier Project\\Merged\\" 

files = sorted(glob.glob(PARQUET_DIR + "*.parquet"))
print(f"Found {len(files)} parquet file(s):")

Found 4 parquet file(s):


In [7]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.spatial import cKDTree

EMBEDDING_COLS = [f'A{i:02d}' for i in range(64)]

def build_neighbour_table(df, patch_size=3, tolerance=0.6):
    """
    Pre-computes a (N, patch_size, patch_size) integer array of neighbour indices.
    Entry is -1 where no neighbour exists (boundary pixels, zero-padded later).
    
    tolerance: fraction of pixel spacing within which we accept a candidate 
               as a true neighbour (0.6 means within 60% of one pixel width).
               This handles slight coordinate irregularities from projected exports.
    """
    half = patch_size // 2
    coords = np.stack([df['lon'].values, df['lat'].values], axis=1)  # (N, 2)
    
    # Estimate pixel spacing (used only for tolerance threshold, not for lookup)
    pixel_lon = np.median(np.diff(np.sort(df['lon'].unique())))
    pixel_lat = np.median(np.diff(np.sort(df['lat'].unique())))
    
    print(f"Building KD-tree for {len(df):,} pixels...")
    tree = cKDTree(coords)
    
    # k = patch_size^2 gives us enough candidates to fill the whole patch
    k = patch_size ** 2 + 1  # +1 to include self
    print(f"Querying {k} nearest neighbours per pixel (this may take a few minutes)...")
    distances, indices = tree.query(coords, k=k, workers=-1)  # workers=-1 = all CPUs
    
    neighbour_table = np.full((len(df), patch_size, patch_size), -1, dtype=np.int32)
    
    for slot in range(k):
        neighbour_coords = coords[indices[:, slot]]           # (N, 2)
        delta_lon = neighbour_coords[:, 0] - coords[:, 0]    # (N,)
        delta_lat = neighbour_coords[:, 1] - coords[:, 1]    # (N,)
        
        # Convert offsets to grid units
        dc = np.round(delta_lon / pixel_lon).astype(int)     # col offset
        dr = np.round(delta_lat / pixel_lat).astype(int)     # row offset
        
        # Check offset is within the patch and within tolerance of a true grid position
        residual_lon = np.abs(delta_lon - dc * pixel_lon)
        residual_lat = np.abs(delta_lat - dr * pixel_lat)
        
        valid = (
            (np.abs(dc) <= half) &
            (np.abs(dr) <= half) &
            (residual_lon < tolerance * pixel_lon) &
            (residual_lat < tolerance * pixel_lat)
        )
        
        patch_row = dr + half
        patch_col = dc + half

        # Clip to valid range before indexing — out-of-bounds slots are excluded
        # by the `valid` mask, but numpy evaluates the index expression first.
        patch_row_safe = np.clip(patch_row, 0, patch_size - 1)
        patch_col_safe = np.clip(patch_col, 0, patch_size - 1)

        pixel_indices = np.arange(len(df))
        mask = valid & (neighbour_table[pixel_indices, patch_row_safe, patch_col_safe] == -1)
        neighbour_table[pixel_indices[mask], patch_row_safe[mask], patch_col_safe[mask]] = \
            indices[mask, slot]
    
    # Diagnostics
    surrounding = neighbour_table.reshape(len(df), patch_size * patch_size)
    centre_idx = half * patch_size + half
    mask_no_centre = np.ones(patch_size * patch_size, dtype=bool)
    mask_no_centre[centre_idx] = False
    all_8_neighbours = np.all(surrounding[:, mask_no_centre] != -1, axis=1)
    zero_neighbours  = np.all(surrounding[:, mask_no_centre] == -1, axis=1)
    
    print(f"\nNeighbour table built:")
    print(f"  Pixels with all 8 neighbours: {all_8_neighbours.mean()*100:.1f}%  "
          f"(expected ~82%)")
    print(f"  Pixels with 0 neighbours:     {zero_neighbours.mean()*100:.1f}%")
    print(f"  Mean neighbours per pixel:    "
          f"{(surrounding[:, mask_no_centre] != -1).sum(axis=1).mean():.2f}")
    
    return neighbour_table, pixel_lon, pixel_lat


class GlacierPatchDataset(Dataset):
    """
    Serves spatial patches of AlphaEarth embeddings centred on each pixel.
    Uses a pre-computed KD-tree neighbour table for robust, fast lookup.
    
    Input tensor shape:  (64, patch_size, patch_size)
    Target:              binary melt_label (float32)
    """
    def __init__(self, parquet_path, patch_size=3, tolerance=0.6):
        assert patch_size % 2 == 1, "patch_size must be odd"
        self.patch_size = patch_size
        self.half = patch_size // 2

        print(f"Loading {parquet_path} ...")
        df = pd.read_parquet(parquet_path)
        print(f"  {len(df):,} pixels")

        self.embeddings = df[EMBEDDING_COLS].values.astype(np.float32)  # (N, 64)
        self.labels     = df['melt_label'].values.astype(np.float32)

        self.neighbour_table, _, _ = build_neighbour_table(df, patch_size, tolerance)
        # neighbour_table: (N, patch_size, patch_size) int32, -1 = missing

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        patch = np.zeros((64, self.patch_size, self.patch_size), dtype=np.float32)
        slots = self.neighbour_table[idx]  # (patch_size, patch_size)

        for i in range(self.patch_size):
            for j in range(self.patch_size):
                n_idx = slots[i, j]
                if n_idx != -1:
                    patch[:, i, j] = self.embeddings[n_idx]
                # else: zero padding for boundary pixels

        return torch.from_numpy(patch), torch.tensor(self.labels[idx])

In [8]:
dataset = GlacierPatchDataset(
    parquet_path=f"{PARQUET_DIR}r1a_combined.parquet",
    patch_size=3
)

patch, label = dataset[0]
print(f"\nSingle sample:")
print(f"  Patch shape: {patch.shape}")
print(f"  Label:       {label.item()}")
print(f"  Centre (A00-A04): {patch[:5, 1, 1]}")

loader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=0)
patches, labels = next(iter(loader))
print(f"\nFirst batch: patches={patches.shape}, labels={labels.shape}")
print(f"Melt rate: {labels.mean():.3f}")

Loading C:\Users\admin\Documents\Glacier Project\Merged\r1a_combined.parquet ...
  6,222,913 pixels
Building KD-tree for 6,222,913 pixels...
Querying 10 nearest neighbours per pixel (this may take a few minutes)...

Neighbour table built:
  Pixels with all 8 neighbours: 92.6%  (expected ~82%)
  Pixels with 0 neighbours:     0.0%
  Mean neighbours per pixel:    7.81

Single sample:
  Patch shape: torch.Size([64, 3, 3])
  Label:       1.0
  Centre (A00-A04): tensor([-0.0222, -0.1034, -0.1477,  0.0121, -0.1663])

First batch: patches=torch.Size([512, 64, 3, 3]), labels=torch.Size([512])
Melt rate: 0.164


In [9]:
import torch
import torch.nn as nn

class GlacierCNN(nn.Module):
    """
    Patch-based CNN for per-pixel glacier melt prediction.
    
    Input:  (B, 64, 3, 3) — 64-channel AlphaEarth embedding patch
    Output: (B, 1)        — raw logit (apply sigmoid for probability)
    
    Architecture: two Conv2d layers with 2×2 kernels progressively 
    reduce the 3×3 spatial dims to 1×1, then FC layers classify.
    We use 2×2 kernels (not 3×3) so we get two conv stages rather 
    than collapsing to 1×1 in one step — this lets the network learn
    hierarchical spatial features rather than just a weighted sum.
    """
    def __init__(self, dropout=0.3):
        super().__init__()
        
        self.conv_block = nn.Sequential(
            # (B, 64, 3, 3) -> (B, 128, 2, 2)
            nn.Conv2d(64, 128, kernel_size=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # (B, 128, 2, 2) -> (B, 256, 1, 1)
            nn.Conv2d(128, 256, kernel_size=2),
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )
        
        self.fc_block = nn.Sequential(
            nn.Flatten(),                        # (B, 256)
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)                     # raw logit
        )
    
    def forward(self, x):
        return self.fc_block(self.conv_block(x)).squeeze(1)

# Quick architecture check
model = GlacierCNN()
dummy = torch.zeros(8, 64, 3, 3)
out = model(dummy)
print(f"Output shape: {out.shape}")     # should be torch.Size([8])
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

Output shape: torch.Size([8])
Trainable parameters: 202,049


In [12]:
from torch.utils.data import DataLoader, random_split, ConcatDataset
from sklearn.metrics import roc_auc_score
import numpy as np

REGIONS = ["r1a", "r1b", "r2", "r3"]

# Load all regions — this will take a few minutes per region
datasets = {}
for region in REGIONS:
    print(f"\n--- Loading {region} ---")
    datasets[region] = GlacierPatchDataset(
        parquet_path=f"{PARQUET_DIR}/{region}_combined.parquet",
        patch_size=3
    )


--- Loading r1a ---
Loading C:\Users\admin\Documents\Glacier Project\Merged\/r1a_combined.parquet ...
  6,222,913 pixels
Building KD-tree for 6,222,913 pixels...
Querying 10 nearest neighbours per pixel (this may take a few minutes)...

Neighbour table built:
  Pixels with all 8 neighbours: 92.6%  (expected ~82%)
  Pixels with 0 neighbours:     0.0%
  Mean neighbours per pixel:    7.81

--- Loading r1b ---
Loading C:\Users\admin\Documents\Glacier Project\Merged\/r1b_combined.parquet ...
  1,028,089 pixels
Building KD-tree for 1,028,089 pixels...
Querying 10 nearest neighbours per pixel (this may take a few minutes)...

Neighbour table built:
  Pixels with all 8 neighbours: 84.3%  (expected ~82%)
  Pixels with 0 neighbours:     0.0%
  Mean neighbours per pixel:    7.57

--- Loading r2 ---
Loading C:\Users\admin\Documents\Glacier Project\Merged\/r2_combined.parquet ...
  6,217,318 pixels
Building KD-tree for 6,217,318 pixels...
Querying 10 nearest neighbours per pixel (this may take a f

In [14]:
# Spatial validation: train on R1a, R1b, R2 — validate on R3
# This is stricter and more meaningful than a random pixel split,
# since a random split leaks spatial autocorrelation into validation
# (a pixel's label is correlated with its neighbours, which may be in train set)
train_dataset = ConcatDataset([datasets["r1a"], datasets["r1b"], datasets["r2"]])
val_dataset   = datasets["r3"]

print(f"Train pixels: {len(train_dataset):,}")
print(f"Val pixels:   {len(val_dataset):,}")

# Class imbalance: weight positive class (melt) inversely to its frequency
# so the loss function treats melt and non-melt pixels equally
all_labels = np.concatenate([
    datasets[r].labels for r in ["r1a", "r1b", "r2"]
])
melt_rate  = all_labels.mean()
pos_weight = torch.tensor((1 - melt_rate) / melt_rate, dtype=torch.float32)
print(f"\nTraining melt rate: {melt_rate:.3f}")
print(f"pos_weight for BCEWithLogitsLoss: {pos_weight:.2f}")
# e.g. if melt_rate=0.16, pos_weight≈5.25 — penalises missed melt pixels 5× more

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=2048, shuffle=False, num_workers=4)

Train pixels: 13,468,320
Val pixels:   1,774,319

Training melt rate: 0.189
pos_weight for BCEWithLogitsLoss: 4.29


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model     = GlacierCNN(dropout=0.3).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)
# Scheduler reduces LR by half if val AUC stops improving for 2 epochs

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    all_labels, all_probs = [], []
    
    with torch.set_grad_enabled(train):
        for patches, labels in loader:
            patches = patches.to(device)
            labels  = labels.to(device)
            
            logits = model(patches)
            loss   = criterion(logits, labels)
            
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            total_loss += loss.item() * len(labels)
            all_probs.extend(torch.sigmoid(logits).cpu().detach().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    auc      = roc_auc_score(all_labels, all_probs)
    # AUC-ROC is the right metric here: it's threshold-independent and 
    # handles class imbalance properly, unlike raw accuracy
    return avg_loss, auc

N_EPOCHS = 20
best_val_auc = 0
best_epoch   = 0
prev_lr      = optimizer.param_groups[0]['lr']

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train AUC':>9}  {'Val Loss':>8}  {'Val AUC':>7}")
print("-" * 55)

for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_auc = run_epoch(train_loader, train=True)
    val_loss,   val_auc   = run_epoch(val_loader,   train=False)
    
    scheduler.step(val_auc)

    # Print LR reduction manually (verbose=True was removed in PyTorch 2.2)
    cur_lr = optimizer.param_groups[0]['lr']
    lr_msg = f"  [LR → {cur_lr:.2e}]" if cur_lr != prev_lr else ""
    prev_lr = cur_lr
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch   = epoch
        torch.save(model.state_dict(), "best_glacier_cnn.pt")
    
    print(f"{epoch:>5}  {train_loss:>10.4f}  {train_auc:>9.4f}  "
          f"{val_loss:>8.4f}  {val_auc:>7.4f}"
          + (" ← best" if epoch == best_epoch else "")
          + lr_msg)

print(f"\nBest val AUC: {best_val_auc:.4f} at epoch {best_epoch}")

Using device: cpu
Epoch  Train Loss  Train AUC  Val Loss  Val AUC
-------------------------------------------------------
